# Four-way K-fold Hybrid Comparison

This notebook creates the exact plot requested for presentation:

1. LSTM/MCLDNN k-fold
2. Normal attention k-fold
3. Differential attention k-fold
4. Automated SNR-aware hybrid k-fold

Important:

- Existing k-fold checkpoints are reused if already present.
- Missing k-fold checkpoints are trained automatically.
- The hybrid is automated using fold-validation accuracy per SNR.
- No alpha blending.
- No monotonic smoothing.
- Final plot is mean ? std across folds.

Output folder:

```text
experiments/5class_fourway_kfold_hybrid/
```

In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import Image, display, FileLink

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_5class.pkl'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

BASELINE_KFOLD_DIR = Path('experiments/5class_baseline_kfold/kfold')
NORMAL_KFOLD_DIR = Path('experiments/5class_attention_kfold/kfold')
DIFF_KFOLD_DIR = Path('experiments/5class_diffattention_kfold/kfold')
OUT_DIR = Path('experiments/5class_fourway_kfold_hybrid')

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())

In [ ]:
# CELL 2: Check required source files
required = [
    'src/train_kfold.py',
    'src/evaluate_fourway_kfold_hybrid.py',
    'src/models/mcldnn.py',
    'src/models/mcldnn_attention.py',
    'src/models/mcldnn_diffattention.py',
    'configs/exp_5class_baseline_kfold.yaml',
    'configs/exp_5class_attention_kfold.yaml',
    'configs/exp_5class_diffattention_kfold.yaml',
]

for f in required:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

def kfold_ready(kfold_dir, n_folds=5):
    return all((kfold_dir / f'fold_{i}/best_model.weights.h5').exists() for i in range(n_folds))

print('LSTM baseline k-fold ready:', kfold_ready(BASELINE_KFOLD_DIR))
print('Normal attention k-fold ready:', kfold_ready(NORMAL_KFOLD_DIR))
print('Diff attention k-fold ready:', kfold_ready(DIFF_KFOLD_DIR))

In [ ]:
# CELL 3: Train missing k-fold models
# Existing complete k-fold folders are reused. Only missing fold sets are trained.

def run_stream(cmd):
    print('Running:', ' '.join(map(str, cmd)))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, process.args)

jobs = [
    ('lstm_baseline_kfold', BASELINE_KFOLD_DIR, 'configs/exp_5class_baseline_kfold.yaml'),
    ('normal_attention_kfold', NORMAL_KFOLD_DIR, 'configs/exp_5class_attention_kfold.yaml'),
    ('diff_attention_kfold', DIFF_KFOLD_DIR, 'configs/exp_5class_diffattention_kfold.yaml'),
]

for name, kfold_dir, cfg in jobs:
    if kfold_ready(kfold_dir):
        print(f'{name}: all 5 fold checkpoints found, skipping training.')
    else:
        print(f'{name}: missing fold checkpoint(s), training now...')
        run_stream([sys.executable, '-u', 'src/train_kfold.py', '--config', cfg, '--datasetpath', str(DATASET)])
        assert kfold_ready(kfold_dir), f'K-fold training finished but checkpoints are missing: {kfold_dir}'

In [ ]:
# CELL 4: Run four-way k-fold hybrid evaluation
if OUT_DIR.exists():
    print('Removing old four-way outputs:', OUT_DIR)
    shutil.rmtree(OUT_DIR)

cmd = [
    sys.executable, '-u', 'src/evaluate_fourway_kfold_hybrid.py',
    '--datasetpath', str(DATASET),
    '--baseline-kfold-dir', str(BASELINE_KFOLD_DIR),
    '--normal-kfold-dir', str(NORMAL_KFOLD_DIR),
    '--diff-kfold-dir', str(DIFF_KFOLD_DIR),
    '--output-dir', str(OUT_DIR),
    '--n-folds', '5',
    '--low-snr-max', '2',
    '--min-val-delta', '0.0',
]
run_stream(cmd)

assert (OUT_DIR / 'results/fourway_kfold_mean_std_acc_per_snr.csv').exists()
assert (OUT_DIR / 'figures/fourway_kfold_lstm_normal_diff_hybrid_acc_vs_snr.png').exists()
print('Four-way k-fold hybrid evaluation complete.')

In [ ]:
# CELL 5: Display result tables
summary = pd.read_csv(OUT_DIR / 'results/fourway_kfold_mean_std_acc_per_snr.csv')
fold_rows = pd.read_csv(OUT_DIR / 'results/fourway_kfold_fold_acc_per_snr.csv')
metadata = pd.read_json(OUT_DIR / 'results/fourway_kfold_metadata.json', typ='series')

print('Mean ? std across folds per SNR')
display(summary)

print('Per-fold values and hybrid route choices')
display(fold_rows)

print('Metadata')
display(metadata)

In [ ]:
# CELL 6: Display final four-way plots
figs = [
    OUT_DIR / 'figures/fourway_kfold_lstm_normal_diff_hybrid_acc_vs_snr.png',
    OUT_DIR / 'figures/automated_hybrid_kfold_delta_vs_models.png',
]

for fig in figs:
    print(fig)
    display(Image(filename=str(fig)))

In [ ]:
# CELL 7: Create repo-ready zip for four-way k-fold results only
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = Path('/kaggle/working') / f'fourway_kfold_hybrid_repo_ready_{stamp}'
zip_path = shutil.make_archive(
    str(zip_base),
    'zip',
    root_dir=str(WORK_DIR),
    base_dir='experiments/5class_fourway_kfold_hybrid',
)

print('Created repo-ready zip:', zip_path)
print('Extract this at repo root. It will create/update:')
print('  experiments/5class_fourway_kfold_hybrid/')
display(FileLink(zip_path))